# Leaf Image Processing

Comparação de filtros no conjunto Apple / PlantVillage, com uma fotografia de cada classe.

**Organização:** a preparação técnica está concentrada no início. A apresentação começa em **Resultados do experimento**, seguida das imagens por filtro e da escolha para a próxima etapa.


## Preparação técnica

Os quatro filtros foram comparados por busca em grades de parâmetros no Experimento 1.
Usamos os **melhores parâmetros encontrados nas grades avaliadas**, sem repetir a busca aqui.

Esta versão contém uma fotografia dos resultados salvos em
`outputs/denoising_output/experimento1_compacto_40_bm3d3/best_params_tuning.json` (arquivo local opcional),
lidos em **21/09/2026, 16:39 UTC**. Sigma 5 e 10 estão completos; sigma 15 está parcial.
Os resultados de tuning não são uma avaliação final de generalização.

O filtro Gaussiano é uma comparação adicional: seus parâmetros são manuais e ele não
participou da busca. Nas células abaixo estão todas as configurações, funções e cálculos.
Execute com o kernel **Python (Leaf Image Processing)**. O snapshot abaixo e as quatro amostras acompanham o repositorio.


In [ ]:
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from bm3d import bm3d, BM3DStages

# Configuracao da apresentacao.
SIGMA_PARAMETROS = 10
FILTRO_VISUAL = None  # Preencha depois: "median", "bilateral", "nlm", "bm3d" ou "gaussian".
JUSTIFICATIVA_VISUAL = ""
FILTRO_FINAL = None  # None usa provisoriamente o vencedor numerico.
JUSTIFICATIVA_FINAL = ""

# A demonstracao aplica os filtros diretamente nas imagens originais.
# sigma do experimento (intensidade 0..255) != sigma espacial do filtro Gaussiano.
PARAMETROS_GAUSSIANO = {"kernel": 5, "sigma": 1.0}

PASTA_PROJETO = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / "notebooks" / "apple_leaf_analysis.ipynb").is_file()),
    None,
)
if PASTA_PROJETO is None:
    raise FileNotFoundError("Execute a partir da raiz do projeto ou da pasta notebooks/.")

PASTA_IMAGENS = PASTA_PROJETO / "data" / "samples"
FONTE_RESULTADOS = "outputs/denoising_output/experimento1_compacto_40_bm3d3/best_params_tuning.json"
DATA_SNAPSHOT = "2026-09-21 16:39 UTC"
CLASSES = {
    "Apple___healthy": "Maçã saudável",
    "Apple___Apple_scab": "Sarna da macieira",
    "Apple___Black_rot": "Podridão negra",
    "Apple___Cedar_apple_rust": "Ferrugem da macieira",
}
# Quarta imagem de cada classe na ordenacao do experimento: fora do tuning.
ARQUIVOS_AMOSTRA = {
    "Apple___healthy": "image (1000).JPG",
    "Apple___Apple_scab": "image (101).JPG",
    "Apple___Black_rot": "image (101).JPG",
    "Apple___Cedar_apple_rust": "image (101).JPG",
}
NOMES_FILTROS = {
    "median": "Mediana",
    "bilateral": "Bilateral",
    "nlm": "Non-Local Means",
    "bm3d": "BM3D",
    "gaussian": "Gaussiano",
}
METODOS_EXPERIMENTO = ("median", "bilateral", "nlm", "bm3d")


In [ ]:
# Resultados congelados: nao consulta nem modifica o experimento em andamento.
RESULTADOS_EXPERIMENTO = {
    "5": {
        "median": {
            "params": {
                "kernel": 3
            },
            "metrics": {
                "mse": 145.62908172607422,
                "psnr": 29.015746691694503,
                "ssim": 0.7108893684753909
            },
            "search_time_seconds": 6.257430794008542,
            "signature": "1a225efce9146ff9090ade9a74998b4c9739b974f33c2ec2bd0c4cfa9c6e8223",
            "num_candidates": 5,
            "num_tuning_samples": 36
        },
        "bilateral": {
            "params": {
                "diameter": 11,
                "sigma_color": 15,
                "sigma_space": 5
            },
            "metrics": {
                "mse": 10.814486468279803,
                "psnr": 37.84144902326625,
                "ssim": 0.9531789619777606
            },
            "search_time_seconds": 456.2859037929011,
            "signature": "6057240eeb227ccaa123d712d28053df2e6cbced0b06abdb6a5a21ea478755da",
            "num_candidates": 500,
            "num_tuning_samples": 36
        },
        "nlm": {
            "params": {
                "h": 2,
                "h_color": 2.5,
                "template_window": 7,
                "search_window": 21
            },
            "metrics": {
                "mse": 16.02357030797888,
                "psnr": 36.16426383872067,
                "ssim": 0.9384547720440496
            },
            "search_time_seconds": 200.88340962701477,
            "signature": "5d6e9f7f7b381665aadf646a78ef623b9010f3525dedcfa42d0b20e16041633c",
            "num_candidates": 40,
            "num_tuning_samples": 36
        },
        "bm3d": {
            "params": {
                "sigma_factor": 1,
                "sigma_psd": 0.0196078431372549,
                "profile": "np",
                "stage": "ALL_STAGES"
            },
            "metrics": {
                "mse": 14.45155617042824,
                "psnr": 36.82070636723775,
                "ssim": 0.9477103899952252
            },
            "search_time_seconds": 328.45147896800336,
            "signature": "ff91343647b682fed044adc3ebd56a6544042449e317664c3aa1b70d54438136",
            "num_candidates": 3,
            "num_tuning_samples": 36
        }
    },
    "10": {
        "median": {
            "params": {
                "kernel": 3
            },
            "metrics": {
                "mse": 159.1414667765299,
                "psnr": 28.005936935776358,
                "ssim": 0.6581708157408964
            },
            "search_time_seconds": 3.891623960007564,
            "signature": "ee3c7aef102ba365e2615e10bc5be4841cc0ba2901e24f1febf44a98ac7c094e",
            "num_candidates": 5,
            "num_tuning_samples": 36
        },
        "bilateral": {
            "params": {
                "diameter": 11,
                "sigma_color": 30,
                "sigma_space": 5
            },
            "metrics": {
                "mse": 30.270472208658855,
                "psnr": 33.470537914729896,
                "ssim": 0.8809276585682988
            },
            "search_time_seconds": 403.029068967051,
            "signature": "6fe5702f15b2aadb3020ddc5b4ada497bc76a8f1f61033ffa54f48ad8d3ff124",
            "num_candidates": 500,
            "num_tuning_samples": 36
        },
        "nlm": {
            "params": {
                "h": 4,
                "h_color": 5,
                "template_window": 7,
                "search_window": 21
            },
            "metrics": {
                "mse": 42.6248313056098,
                "psnr": 32.08203998142299,
                "ssim": 0.8601418188208684
            },
            "search_time_seconds": 206.25895732900244,
            "signature": "819ed065a6fd7bbe1fcadd1aac0d35b150adf77ab20953c2082a8e0fc2289416",
            "num_candidates": 40,
            "num_tuning_samples": 36
        },
        "bm3d": {
            "params": {
                "sigma_factor": 1,
                "sigma_psd": 0.0392156862745098,
                "profile": "np",
                "stage": "ALL_STAGES"
            },
            "metrics": {
                "mse": 41.00092160260236,
                "psnr": 32.553789595803806,
                "ssim": 0.8604389570931061
            },
            "search_time_seconds": 334.8361483100016,
            "signature": "89f5f210c423a79aa0ddcf15d477ccf5f51e2aea958d4b9c44f3cc0abe0bf8e2",
            "num_candidates": 3,
            "num_tuning_samples": 36
        }
    },
    "15": {
        "median": {
            "params": {
                "kernel": 5
            },
            "metrics": {
                "mse": 188.54212315877282,
                "psnr": 26.994856309196038,
                "ssim": 0.5793776052173629
            },
            "search_time_seconds": 3.876872020002338,
            "signature": "e053b94f41eada37865edd9524b2c101e1a848be5e4bd69f58427e78a6413e44",
            "num_candidates": 5,
            "num_tuning_samples": 36
        },
        "bilateral": {
            "params": {
                "diameter": 11,
                "sigma_color": 50,
                "sigma_space": 5
            },
            "metrics": {
                "mse": 53.63194868299696,
                "psnr": 31.128840537135154,
                "ssim": 0.8095448902264134
            },
            "search_time_seconds": 400.19017765404715,
            "signature": "f199f2789d35578b2e41435780e2f287184f6cf193da6fc963c05946624979a4",
            "num_candidates": 500,
            "num_tuning_samples": 36
        }
    }
}


### Fórmulas e critérios

Para a imagem de referência \(I\) e a filtrada \(\hat I\), o experimento usa:

\[
MSE=\frac{1}{N}\sum_{i=1}^{N}(I_i-\hat I_i)^2,\qquad
PSNR=10\log_{10}\left(\frac{255^2}{MSE}\right).
\]

O PSNR médio no tuning determina a escolha numérica; SSIM é o desempate.
Menor MSE e maiores PSNR/SSIM são desejáveis. Não recalculamos essas métricas contra
as originais desta demonstração, pois não há referência limpa independente para seu ruído real.

O filtro Gaussiano usa pesos espaciais proporcionais a
\[
G(x,y)=\exp\left(-\frac{x^2+y^2}{2\sigma_g^2}\right),
\]
normalizados na janela \(5\times5\), com \(\sigma_g=1\) pixel.
Já o BM3D recebe o desvio-padrão de intensidade normalizado: \(\sigma_{\rm BM3D}=\sigma_{\rm ruído}/255\) quando o fator selecionado é 1.

A inspeção visual considera preservação de lesões, bordas e textura, alteração de cor
e suavização excessiva. A escolha visual só é registrada após avaliação humana.


In [ ]:
def preparar_resultados():
    completos = [
        int(sigma) for sigma, methods in RESULTADOS_EXPERIMENTO.items()
        if all(method in methods for method in METODOS_EXPERIMENTO)
    ]
    if SIGMA_PARAMETROS not in completos:
        raise ValueError(f"Escolha um sigma completo: {sorted(completos)}.")
    registro = RESULTADOS_EXPERIMENTO[str(SIGMA_PARAMETROS)]
    parametros = {m: registro[m]["params"].copy() for m in METODOS_EXPERIMENTO}
    parametros["gaussian"] = PARAMETROS_GAUSSIANO.copy()
    vencedor = max(METODOS_EXPERIMENTO, key=lambda m: (
        registro[m]["metrics"]["psnr"], registro[m]["metrics"]["ssim"]))
    for selected in (FILTRO_VISUAL, FILTRO_FINAL):
        if selected is not None and selected not in NOMES_FILTROS:
            raise ValueError(f"Filtro desconhecido: {selected}")
    return parametros, vencedor


def carregar_amostras():
    imagens = {}
    for classe in CLASSES:
        caminho = PASTA_IMAGENS / classe / ARQUIVOS_AMOSTRA[classe]
        imagem = cv2.imread(str(caminho), cv2.IMREAD_COLOR)
        if imagem is None:
            raise FileNotFoundError(f"Nao foi possivel ler: {caminho}")
        imagens[classe] = imagem
    return imagens


def aplicar_filtro(imagem, metodo, parametros=None):
    """Aplica um filtro a uma imagem BGR uint8; utilizavel na proxima etapa."""
    if imagem.dtype != np.uint8 or imagem.ndim != 3 or imagem.shape[2] != 3:
        raise ValueError("Use uma imagem BGR uint8 com tres canais.")
    p = PARAMETROS[metodo] if parametros is None else parametros
    if metodo == "median":
        return cv2.medianBlur(imagem, int(p["kernel"]))
    if metodo == "bilateral":
        return cv2.bilateralFilter(imagem, int(p["diameter"]),
                                   float(p["sigma_color"]), float(p["sigma_space"]))
    if metodo == "nlm":
        return cv2.fastNlMeansDenoisingColored(
            imagem, None, h=float(p["h"]), hColor=float(p["h_color"]),
            templateWindowSize=int(p["template_window"]),
            searchWindowSize=int(p["search_window"]))
    if metodo == "bm3d":
        rgb = cv2.cvtColor(imagem, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
        filtrada = bm3d(rgb, sigma_psd=float(p["sigma_psd"]), profile=p["profile"],
                       stage_arg=getattr(BM3DStages, p["stage"]))
        filtrada = (np.clip(filtrada, 0, 1) * 255 + 0.5).astype(np.uint8)
        return cv2.cvtColor(filtrada, cv2.COLOR_RGB2BGR)
    if metodo == "gaussian":
        k = int(p["kernel"])
        if k <= 0 or k % 2 == 0 or float(p["sigma"]) <= 0:
            raise ValueError("O Gaussiano exige janela positiva impar e sigma positivo.")
        return cv2.GaussianBlur(imagem, (k, k), sigmaX=float(p["sigma"]),
                                sigmaY=float(p["sigma"]))
    raise ValueError(f"Filtro desconhecido: {metodo}")


def resultados_visuais(metodo):
    if metodo not in CACHE_FILTROS:
        CACHE_FILTROS[metodo] = {
            classe: aplicar_filtro(imagem, metodo) for classe, imagem in AMOSTRAS.items()
        }
    return CACHE_FILTROS[metodo]


def desenhar_imagem(ax, imagem, titulo):
    ax.imshow(cv2.cvtColor(imagem, cv2.COLOR_BGR2RGB), interpolation="nearest")
    ax.set_title(titulo, fontsize=11)
    ax.axis("off")


def mostrar_linha(imagens, titulo):
    fig, axes = plt.subplots(1, len(CLASSES), figsize=(16, 4.6), layout="constrained")
    for ax, classe in zip(axes, CLASSES):
        desenhar_imagem(ax, imagens[classe], CLASSES[classe])
    fig.suptitle(titulo, fontsize=16)
    plt.show()
    plt.close(fig)


def mostrar_originais():
    mostrar_linha(AMOSTRAS, "Referência visual — imagens originais")


def mostrar_filtro(metodo):
    mostrar_linha(resultados_visuais(metodo), NOMES_FILTROS[metodo])


def mostrar_resultados_experimento():
    status = []
    linhas = []
    for sigma, methods in sorted(RESULTADOS_EXPERIMENTO.items(), key=lambda item: int(item[0])):
        completo = all(m in methods for m in METODOS_EXPERIMENTO)
        status.append({"Sigma": int(sigma), "Filtros concluídos": len(methods),
                       "Situação": "Completo" if completo else "Parcial; fora da seleção"})
        if completo:
            for metodo in METODOS_EXPERIMENTO:
                data = methods[metodo]
                linhas.append({
                    "Sigma": int(sigma), "Filtro": NOMES_FILTROS[metodo],
                    "PSNR (dB)": data["metrics"]["psnr"], "SSIM": data["metrics"]["ssim"],
                    "MSE": data["metrics"]["mse"], "Candidatos": data["num_candidates"],
                    "Amostras de tuning": data["num_tuning_samples"],
                })
    display(pd.DataFrame(status).style.hide(axis="index"))
    tabela = pd.DataFrame(linhas).sort_values(["Sigma", "PSNR (dB)"], ascending=[True, False])
    display(tabela.style.hide(axis="index").format(
        {"PSNR (dB)": "{:.3f}", "SSIM": "{:.4f}", "MSE": "{:.3f}"}))
    display(Markdown(
        f"**Escolha numérica para sigma {SIGMA_PARAMETROS}: "
        f"{NOMES_FILTROS[FILTRO_NUMERICO]}.** "
        "Resultados médios de tuning; não são métricas das quatro fotos exibidas abaixo. "
        "Gaussiano não participou desta busca."))


def mostrar_parametros():
    linhas = []
    for metodo, p in PARAMETROS.items():
        descricao = ", ".join(f"{k}={v:.6g}" if isinstance(v, float) else f"{k}={v}"
                              for k, v in p.items())
        linhas.append({"Filtro": NOMES_FILTROS[metodo], "Parâmetros": descricao,
                       "Origem": "Manual; comparação adicional" if metodo == "gaussian"
                       else f"Melhor da grade no tuning; sigma {SIGMA_PARAMETROS}"})
    display(pd.DataFrame(linhas).style.hide(axis="index"))


def mostrar_amostras():
    display(pd.DataFrame([
        {"Classe": CLASSES[c], "Pasta": c, "Arquivo": ARQUIVOS_AMOSTRA[c]}
        for c in CLASSES
    ]).style.hide(axis="index"))
    mostrar_originais()


def mostrar_escolha():
    escolhido = FILTRO_FINAL or FILTRO_NUMERICO
    valores = RESULTADOS_EXPERIMENTO[str(SIGMA_PARAMETROS)][FILTRO_NUMERICO]["metrics"]
    resumo = (f"**Melhor pelas métricas:** {NOMES_FILTROS[FILTRO_NUMERICO]} "
              f"(sigma {SIGMA_PARAMETROS}, PSNR médio {valores['psnr']:.3f} dB, "
              f"SSIM {valores['ssim']:.4f}).")
    if FILTRO_VISUAL is None:
        resumo += "\n\n**Escolha visual:** ainda não informada."
    else:
        motivo = JUSTIFICATIVA_VISUAL.strip() or "Justificativa visual ainda não preenchida."
        resumo += f"\n\n**Escolha visual:** {NOMES_FILTROS[FILTRO_VISUAL]}. {motivo}"
    if FILTRO_FINAL is None:
        motivo = ("Seleção provisória baseada no maior PSNR médio do tuning. "
                  "A decisão visual ainda deve ser considerada para o pipeline.")
    else:
        motivo = JUSTIFICATIVA_FINAL.strip() or "Justificativa final ainda não preenchida."
    resumo += f"\n\n**Filtro para a próxima etapa:** {NOMES_FILTROS[escolhido]}. {motivo}"
    display(Markdown(resumo))
    filtradas = resultados_visuais(escolhido)
    fig, axes = plt.subplots(2, len(CLASSES), figsize=(16, 8), layout="constrained")
    for coluna, classe in enumerate(CLASSES):
        desenhar_imagem(axes[0, coluna], AMOSTRAS[classe], f"{CLASSES[classe]}\nOriginal")
        desenhar_imagem(axes[1, coluna], filtradas[classe],
                       f"{CLASSES[classe]}\n{NOMES_FILTROS[escolhido]}")
    fig.suptitle("Filtro selecionado — comparação com as originais", fontsize=16)
    plt.show()
    plt.close(fig)


PARAMETROS, FILTRO_NUMERICO = preparar_resultados()
AMOSTRAS = carregar_amostras()
CACHE_FILTROS = {}


### FFT, reconstrução e bordas

A Fase A termina com \(F=\mathrm{fftshift}(\mathrm{FFT2}(I))\),
\(G=H F\) e \(I_f=\Re[\mathrm{IFFT2}(\mathrm{ifftshift}(G))]\), por canal.
Para inspeção, usamos luminância sem média, janela Hann e amplitude em dB;
essa janela **não** entra na reconstrução. A escala dos espectros é compartilhada.

O candidato passa-baixas usa \(H(f_x,f_y)=\exp[-(f_x^2+f_y^2)/(2 f_c^2)]\).
O notch opcional rejeita pares conjugados de picos indicados manualmente, preservando DC.
A FFT assume extensão periódica: a filtragem pode introduzir artefatos nas bordas.
Não confundir erro numérico de reconstrução com PSNR/SSIM de qualidade.

Na Fase B, \(G_x=S_x*I,\ G_y=S_y*I,\ M=\sqrt{G_x^2+G_y^2}\).
Sobel mostra \(M\); para comparar mapas, usamos um limiar fixo.
Canny usa supressão de não máximos e histerese com dois limiares fixos.
Ambos recebem a mesma imagem pré-processada, em cinza; Canny não recebe o mapa Sobel.

Estabilidade: cinco perturbações gaussianas independentes na saída da Fase A.
\(\mathrm{IoU}=|A\cap B|/|A\cup B|\).
O F1 tolerante combina a fração de pontos de B próximos de A e vice-versa,
com tolerância de um pixel em cada eixo. Não há contorno verdadeiro anotado.

Referências: [NumPy FFT](https://numpy.org/doc/stable/reference/routines.fft.html),
[OpenCV Sobel](https://docs.opencv.org/4.13.0/d2/d2c/tutorial_sobel_derivatives.html),
[OpenCV Canny](https://docs.opencv.org/4.10.0/da/d22/tutorial_py_canny.html).


In [ ]:
# FFT: frequencias em ciclos/pixel. Inspecao nao altera o sinal reconstruido.
JANELA_HANN_ESPECTRO = True
CORTE_GAUSSIANO_FFT = 0.20  # Candidato manual para comparacao; nao e otimo da busca.
APLICAR_FILTRO_FREQUENCIAL = False  # Ativa notch apenas com picos e justificativa.
JUSTIFICATIVA_FREQUENCIAL = ""
PICOS_NOTCH = {classe: [] for classe in CLASSES}  # Pares (fx, fy); conjugados automaticos.
LARGURA_NOTCH = 0.015

# Bordas: parametros manuais fixos para todas as classes e perturbacoes.
LIMIAR_SOBEL = 100.0
LIMIARES_CANNY = (50.0, 100.0)
SIGMA_ESTABILIDADE = 2.0  # Intensidades 0..255, depois do pre-processamento.
REPETICOES_ESTABILIDADE = 5
SEMENTE_BORDAS = 42
TOLERANCIA_BORDAS = 1


In [ ]:
def fft_centrada(imagem):
    return np.fft.fftshift(np.fft.fft2(imagem, axes=(0, 1), norm="ortho"), axes=(0, 1))


def ifft_centrada(espectro):
    return np.fft.ifft2(np.fft.ifftshift(espectro, axes=(0, 1)), axes=(0, 1), norm="ortho")


def mascara_notch(formato, picos, largura):
    if not np.isfinite(largura) or largura <= 0:
        raise ValueError("Largura deve ser positiva e finita.")
    h, w = formato[:2]
    yy, xx = np.meshgrid(np.fft.fftshift(np.fft.fftfreq(h)),
                         np.fft.fftshift(np.fft.fftfreq(w)), indexing="ij")
    mascara = np.ones((h, w))
    for px, py in picos:
        if not np.isfinite([px, py]).all() or not (-.5 <= px < .5 and -.5 <= py < .5):
            raise ValueError("Picos devem estar em [-0.5, 0.5) ciclos/pixel.")
        if np.hypot(px, py) <= largura:
            raise ValueError("Nao remova a componente DC.")
        for sinal in (-1, 1):
            dx = (xx - sinal * px + .5) % 1 - .5
            dy = (yy - sinal * py + .5) % 1 - .5
            mascara *= -np.expm1(-(dx**2 + dy**2) / (2 * largura**2))
    mascara[h // 2, w // 2] = 1
    return mascara


def processar_frequencias(imagem, picos=(), largura=.015, corte=None):
    entrada = imagem.astype(np.float64) / 255
    espectro = fft_centrada(entrada)
    mascara = mascara_notch(imagem.shape, picos, largura)
    if corte is not None:
        if not np.isfinite(corte) or not 0 < corte <= .5:
            raise ValueError("Corte deve estar em (0, 0.5] ciclos/pixel.")
        fy = np.fft.fftshift(np.fft.fftfreq(imagem.shape[0]))[:, None]
        fx = np.fft.fftshift(np.fft.fftfreq(imagem.shape[1]))[None, :]
        mascara *= np.exp(-(fx**2 + fy**2) / (2 * corte**2))
    reconstruida = ifft_centrada(espectro * mascara[..., None])
    residuo = float(np.max(np.abs(reconstruida.imag)))
    if residuo > 1e-10:
        raise ValueError("Mascara sem simetria conjugada.")
    return {
        "entrada": imagem, "saida": np.rint(np.clip(reconstruida.real, 0, 1) * 255).astype(np.uint8),
        "mascara": mascara,
        "erro_roundtrip": float(np.max(np.abs(ifft_centrada(espectro).real - entrada)) * 255),
        "residuo_imaginario": residuo,
    }


def espectro_para_inspecao(imagem):
    cinza = cv2.cvtColor(imagem, cv2.COLOR_BGR2GRAY).astype(float) / 255
    cinza -= cinza.mean()
    if JANELA_HANN_ESPECTRO:
        cinza *= np.outer(np.hanning(cinza.shape[0]), np.hanning(cinza.shape[1]))
    return 20 * np.log10(np.maximum(np.abs(fft_centrada(cinza)), 1e-12))


def analisar_fft():
    if APLICAR_FILTRO_FREQUENCIAL and not JUSTIFICATIVA_FREQUENCIAL.strip():
        raise ValueError("Preencha JUSTIFICATIVA_FREQUENCIAL.")
    if APLICAR_FILTRO_FREQUENCIAL and not any(PICOS_NOTCH.values()):
        raise ValueError("Indique picos periodicos para aplicar notch.")
    if set(PICOS_NOTCH) - set(CLASSES):
        raise ValueError("Classe desconhecida em PICOS_NOTCH.")
    metodo = FILTRO_FINAL or FILTRO_NUMERICO
    classes = {}
    for classe, entrada in resultados_visuais(metodo).items():
        picos = PICOS_NOTCH.get(classe, []) if APLICAR_FILTRO_FREQUENCIAL else []
        dados = processar_frequencias(entrada, picos, LARGURA_NOTCH)
        # Candidato ilustrativo sempre calculado, sem adocao automatica.
        dados["candidato"] = processar_frequencias(entrada, corte=CORTE_GAUSSIANO_FFT)
        classes[classe] = dados
    return {"metodo": metodo, "classes": classes, "filtro_ativo": APLICAR_FILTRO_FREQUENCIAL}


def mostrar_espectros(etapa):
    imagens = [AMOSTRAS, {c: d["entrada"] for c, d in etapa["classes"].items()}]
    espectros = [[espectro_para_inspecao(linha[c]) for c in CLASSES] for linha in imagens]
    referencia = max(s.max() for linha in espectros for s in linha)
    fig, axes = plt.subplots(2, len(CLASSES), figsize=(16, 8), layout="constrained")
    for i, nome in enumerate(("Original", "Apos denoising")):
        for j, classe in enumerate(CLASSES):
            m = axes[i, j].imshow(espectros[i][j] - referencia, origin="lower",
                extent=(-.5, .5, -.5, .5), cmap="magma", vmin=-80, vmax=0)
            axes[i, j].set_title(f"{CLASSES[classe]}\n{nome}", fontsize=11)
            axes[i, j].set_xlabel("fx (ciclos/pixel)")
            axes[i, j].set_ylabel("fy (ciclos/pixel)")
    fig.colorbar(m, ax=axes.ravel().tolist(), label="Amplitude (dB), referencia comum", shrink=.7)
    plt.show()
    plt.close(fig)


def mostrar_filtro_frequencial(etapa):
    fig, axes = plt.subplots(4, len(CLASSES), figsize=(16, 13), layout="constrained")
    for j, (classe, d) in enumerate(etapa["classes"].items()):
        candidato = d["candidato"]
        desenhar_imagem(axes[0, j], d["entrada"], CLASSES[classe] + "\nApos denoising")
        axes[1, j].imshow(candidato["mascara"], cmap="gray", vmin=0, vmax=1,
                          origin="lower", extent=(-.5, .5, -.5, .5))
        axes[1, j].set_title("H: passa-baixas Gaussiano", fontsize=11)
        axes[1, j].set_xlabel("fx (ciclos/pixel)")
        axes[1, j].set_ylabel("fy (ciclos/pixel)")
        desenhar_imagem(axes[2, j], candidato["saida"], "IFFT do candidato (nao adotado)")
        desenhar_imagem(axes[3, j], d["saida"], "IFFT selecionada para Fase B")
    plt.show()
    plt.close(fig)
    if etapa["filtro_ativo"]:
        display(Markdown("**Decisao: notch nos picos indicados.** " + JUSTIFICATIVA_FREQUENCIAL))
    else:
        display(Markdown("**Decisao: manter o denoising espacial (H = 1).** "
            "O passa-baixas acima demonstra FFT → H × F → IFFT, mas nao foi otimizado "
            "nem adotado: pode apagar texturas e lesoes. Nao foram anotados picos de ruido "
            "periodico que justifiquem notch. A FFT/IFFT selecionada preserva a entrada."))
    display(pd.DataFrame([
        {"Classe": CLASSES[c], "Erro numerico FFT/IFFT (0..255)": d["erro_roundtrip"],
         "Residuo imaginario": d["residuo_imaginario"],
         "Picos notch aplicados": len(PICOS_NOTCH.get(c, [])) if etapa["filtro_ativo"] else 0}
        for c, d in etapa["classes"].items()
    ]))
    if etapa["filtro_ativo"]:
        fig, axes = plt.subplots(1, len(CLASSES), figsize=(16, 4), layout="constrained")
        for ax, (c, d) in zip(axes, etapa["classes"].items()):
            ax.imshow(d["mascara"], cmap="gray", vmin=0, vmax=1, origin="lower",
                      extent=(-.5, .5, -.5, .5))
            ax.set_title(CLASSES[c])
        plt.show()
        plt.close(fig)


In [ ]:
def detectar_bordas(imagem):
    cinza = cv2.cvtColor(imagem, cv2.COLOR_BGR2GRAY)
    gx = cv2.Sobel(cinza, cv2.CV_32F, 1, 0, ksize=3)
    gy = cv2.Sobel(cinza, cv2.CV_32F, 0, 1, ksize=3)
    magnitude = cv2.magnitude(gx, gy)
    return {
        "sobel": magnitude,
        "sobel_binario": magnitude >= LIMIAR_SOBEL,
        "canny": cv2.Canny(cinza, LIMIARES_CANNY[0], LIMIARES_CANNY[1],
                           apertureSize=3, L2gradient=True) > 0,
    }


def sobreposicao(imagem, bordas):
    rgb = cv2.cvtColor(imagem, cv2.COLOR_BGR2RGB).copy()
    rgb[bordas] = (255, 40, 40)
    return rgb


def mostrar_bordas(imagens, resultados):
    fig, axes = plt.subplots(4, len(CLASSES), figsize=(16, 13), layout="constrained")
    for j, classe in enumerate(CLASSES):
        d = resultados[classe]
        desenhar_imagem(axes[0, j], imagens[classe], CLASSES[classe] + "\nPre-processada")
        axes[1, j].imshow(d["sobel"], cmap="gray", vmin=0, vmax=1020 * np.sqrt(2))
        axes[1, j].set_title("Sobel: magnitude (escala fixa)", fontsize=11)
        axes[2, j].imshow(d["canny"], cmap="gray", vmin=0, vmax=1)
        axes[2, j].set_title("Canny: mapa binario", fontsize=11)
        axes[3, j].imshow(sobreposicao(imagens[classe], d["canny"]))
        axes[3, j].set_title("Canny sobre a imagem", fontsize=11)
        for i in range(1, 4):
            axes[i, j].axis("off")
    plt.show()
    plt.close(fig)


def comparar_mapas(a, b, tolerancia):
    if not isinstance(tolerancia, (int, np.integer)) or tolerancia < 0:
        raise ValueError("Tolerancia deve ser um inteiro nao negativo.")
    a, b = a.astype(bool), b.astype(bool)
    if a.shape != b.shape:
        raise ValueError("Mapas devem ter a mesma forma.")
    uniao = np.count_nonzero(a | b)
    iou = np.count_nonzero(a & b) / uniao if uniao else np.nan
    if not a.any() or not b.any():
        return iou, (0.0 if uniao else np.nan)
    kernel = np.ones((2 * tolerancia + 1,) * 2, np.uint8)
    da = cv2.dilate(a.astype(np.uint8), kernel) > 0
    db = cv2.dilate(b.astype(np.uint8), kernel) > 0
    precisao = np.count_nonzero(b & da) / np.count_nonzero(b)
    revocacao = np.count_nonzero(a & db) / np.count_nonzero(a)
    f1 = 2 * precisao * revocacao / (precisao + revocacao) if precisao + revocacao else 0.0
    return iou, f1


def avaliar_estabilidade(imagens, resultados):
    rng = np.random.default_rng(SEMENTE_BORDAS)
    linhas, perturbadas = [], {}
    for classe, imagem in imagens.items():
        base = resultados[classe]
        medidas = {"sobel_binario": [], "canny": []}
        for repeticao in range(REPETICOES_ESTABILIDADE):
            ruido = rng.normal(0, SIGMA_ESTABILIDADE, imagem.shape)
            variante = np.rint(np.clip(imagem.astype(float) + ruido, 0, 255)).astype(np.uint8)
            bordas = detectar_bordas(variante)
            if repeticao == 0:
                perturbadas[classe] = bordas
            for metodo in medidas:
                medidas[metodo].append(comparar_mapas(base[metodo], bordas[metodo], TOLERANCIA_BORDAS))
        for metodo, valores in medidas.items():
            valores = np.array(valores)
            linhas.append({"Classe": CLASSES[classe],
                "Detector": "Sobel limiarizado" if metodo == "sobel_binario" else "Canny",
                "Bordas (%)": 100 * base[metodo].mean(),
                "IoU medio": valores[:, 0].mean(), "F1 tolerante medio": valores[:, 1].mean(),
                "Desvio F1": valores[:, 1].std()})
    return pd.DataFrame(linhas), perturbadas


def mostrar_estabilidade(imagens, resultados, perturbadas, tabela):
    display(tabela.round(4))
    fig, axes = plt.subplots(2, len(CLASSES), figsize=(16, 7), layout="constrained")
    for j, classe in enumerate(CLASSES):
        for i, metodo in enumerate(("sobel_binario", "canny")):
            a, b = resultados[classe][metodo], perturbadas[classe][metodo]
            rgb = np.zeros((*a.shape, 3), dtype=np.uint8)
            rgb[a & b] = (220, 220, 220)
            rgb[a & ~b] = (255, 70, 70)
            rgb[b & ~a] = (30, 200, 255)
            axes[i, j].imshow(rgb)
            axes[i, j].set_title(CLASSES[classe] + "\n" +
                ("Sobel limiarizado" if i == 0 else "Canny"), fontsize=11)
            axes[i, j].axis("off")
    fig.suptitle("Primeira perturbacao: branco = comum; vermelho = perdido; azul = novo")
    plt.show()
    plt.close(fig)
    display(Markdown("**Leitura:** maior IoU/F1 indica repetibilidade neste teste, nao "
        "acuracia do contorno. O F1 aceita deslocamentos de ate um pixel em cada eixo; "
        "mapas ambos vazios recebem NaN. Densidade e espessura das bordas afetam a comparacao. "
        "A perturbacao e aplicada depois da Fase A: testa os detectores, nao todo o denoising. "
        "A escolha final depende tambem da preservacao visual das lesoes, sem vencedor automatico."))


## Resultados do experimento

Os valores abaixo são os resultados concluídos disponíveis na data desta versão.
A seleção inicial usa **sigma 10**, o maior sigma com os quatro métodos concluídos no snapshot.


In [ ]:
mostrar_resultados_experimento()


## Parâmetros utilizados

Aplicamos diretamente os parâmetros selecionados. Não há busca por parâmetros neste notebook.


In [ ]:
mostrar_parametros()


## Uma imagem por classe

A mesma fotografia de cada classe será usada em todos os filtros, na ordem:
saudável, sarna, podridão negra e ferrugem. São as quartas imagens da ordenação do
experimento, fora das três imagens de cada classe usadas no tuning.

Esta é uma demonstração visual em imagens originais, sem adicionar ruído sintético.
A transferência dos parâmetros obtidos com ruído artificial exige inspeção visual.


In [ ]:
mostrar_amostras()


## Filtro 1 — Mediana


In [ ]:
mostrar_filtro("median")


## Filtro 2 — Bilateral


In [ ]:
mostrar_filtro("bilateral")


## Filtro 3 — Non-Local Means


In [ ]:
mostrar_filtro("nlm")


## Filtro 4 — BM3D


In [ ]:
mostrar_filtro("bm3d")


## Filtro 5 — Gaussiano


In [ ]:
mostrar_filtro("gaussian")


## Escolha para a próxima etapa

A escolha numérica e a escolha visual são registradas separadamente.
A configuração inicial mostra o vencedor numérico como seleção provisória.
O bloco de configuração no início permite registrar a escolha visual, sua justificativa
e a decisão final, sem alterar as células de apresentação.


In [ ]:
mostrar_escolha()


## Fase A — FFT e inspeção do espectro

Comparação da original com a saída do filtro espacial escolhido. Textura e lesões também produzem altas frequências: energia alta não significa necessariamente ruído.


In [ ]:
ETAPA_FFT = analisar_fft()
mostrar_espectros(ETAPA_FFT)


## Fase A — filtro frequencial e IFFT

O passa-baixas é executado como candidato ilustrativo, com parâmetro manual. A saída selecionada usa notch somente se houver picos e justificativa preenchidos no início; caso contrário, preserva o denoising com H = 1. Abaixo ficam explícitos o candidato e a saída que seguirá para as bordas.


In [ ]:
mostrar_filtro_frequencial(ETAPA_FFT)
IMAGENS_PRE_PROCESSADAS = {c: d["saida"].copy() for c, d in ETAPA_FFT["classes"].items()}


## Fase B — Sobel e Canny

Dois ramos independentes sobre a mesma saída da Fase A: magnitude do gradiente com Sobel e mapa binário com Canny. Os parâmetros são manuais, não resultados da busca de denoising.


In [ ]:
RESULTADOS_BORDAS = {c: detectar_bordas(im) for c, im in IMAGENS_PRE_PROCESSADAS.items()}
mostrar_bordas(IMAGENS_PRE_PROCESSADAS, RESULTADOS_BORDAS)


## Fase B — estabilidade dos contornos

Mede repetibilidade sob ruído leve, sem reajustar os limiares por imagem. A sobreposição mostra a primeira perturbação; a tabela resume as cinco. Não mede acurácia nem substitui a inspeção das lesões.


In [ ]:
TABELA_ESTABILIDADE, BORDAS_PERTURBADAS = avaliar_estabilidade(IMAGENS_PRE_PROCESSADAS, RESULTADOS_BORDAS)
mostrar_estabilidade(IMAGENS_PRE_PROCESSADAS, RESULTADOS_BORDAS, BORDAS_PERTURBADAS, TABELA_ESTABILIDADE)


## Saídas do pipeline

**Fase A concluída:** denoising → métricas de tuning já registradas → FFT → comparação frequencial → decisão → IFFT.

**Fase B concluída:** imagem pré-processada → Sobel e Canny → comparação de estabilidade. As imagens, mapas e tabela permanecem em `IMAGENS_PRE_PROCESSADAS`, `RESULTADOS_BORDAS` e `TABELA_ESTABILIDADE` para a próxima etapa. A seleção visual e os parâmetros de bordas continuam sujeitos à avaliação humana.
